# Inbound Auth

AgentCore Identity lets you validate inbound access (Inbound Auth) for users and applications calling agents or tools in an AgentCore Runtime or validate access to AgentCore Gateway targets. It also provide secure outbound access (Outbound Auth) from an agent to external services or a Gateway target. It integrates with your existing identity providers (such as Amazon Cognito) while enforcing permission boundaries for agents acting independently or on behalf of users (via OAuth).

Inbound Auth validates callers attempting to invoke agents or tools, whether they're hosted in AgentCore Runtime, AgentCore Gateway , or in other environments. Inbound Auth works with IAM (SigV4 credentials) or with OAuth authorization.

By default, Amazon Bedrock AgentCore uses IAM credentials, meaning user requests to the agent are authenticated with the user's IAM credentials. If you use OAuth, you will need to specify the following when configuring your AgentCore Runtime resources or AgentCore Gateway endpoints:

- OAuth discovery server Url — A string that must match the pattern ^.+/\.well-known/openid-configuration$ for OpenID Connect discovery URLs

- Allowed audiences — List of allowed audiences for JWT tokens

- Allowed clients — List of allowed client identifiers

If you use the AgentCore CLI, you can specify the type of authorization (and OAuth discovery server) for an AgentCore Runtime when you use the **configure** command. You can also use the CreateAgentRuntime operation and Amazon Bedrock AgentCore console. If you are creating a Gateway, you use the CreateGateway operation, or the console.

Before the user can use the agent, the client application must have the user authenticate with the OAuth authorizer. Your client receives a bearer token which it then passes to the agent in an invocation request. Upon receipt the agent validates the token with the authorization server before allowing access.


## Overview

In this tutorial we will modify the agent you deployed in 01-AgentCore-runtime and configure it for Inbound Auth using Cognito as the Identity provider. You will set up a Cognito User pool with one user and an app client. You will learn how to host your existing agent, using Amazon Bedrock AgentCore Runtime with Inbound Auth using the Cognito user pool. 

### Tutorial Architecture

<div style="text-align:center">
    <img src="images/inbound_auth_cognito.png" width="90%"/>
</div>

### Tutorial Details


| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                                   |
| Agent type          | Single                                                                           |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                      |
| Tutorial components | Hosting agent on AgentCore Runtime. Using Strands Agent and Amazon Bedrock Model |
| Tutorial vertical   | Cross-vertical                                                                   |
| Example complexity  | Easy                                                                             |
| Inbound Auth        | Cognito                                                                          |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                                     |



### Tutorial Key Features

* Hosting Agents on Amazon Bedrock AgentCore Runtime with Inbound Auth using Amazon Cognito
* Using Amazon Bedrock models
* Using Strands Agents


## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker running

In [1]:
!pip install --force-reinstall -U -r requirements.txt --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.83.7 requires aiohttp==3.13.5, but you have aiohttp 3.14.3 which is incompatible.
litellm 1.83.7 requires click==8.1.8, but you have click 8.4.2 which is incompatible.
litellm 1.83.7 requires jsonschema==4.23.0, but you have jsonschema 4.26.0 which is incompatible.
litellm 1.83.7 requires pydantic==2.12.5, but you have pydantic 2.13.4 which is incompatible.
litellm 1.83.7 requires python-dotenv==1.0.1, but you have python-dotenv 1.2.2 which is incompatible.
langgraph-sdk 0.4.2 requires websockets<16,>=14, but you have websockets 17.0.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.43.0 requires opentelemetry-sdk~=1.43.0, but you have opentelemetry-sdk 1.44.0 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade 

In [13]:
invoke_response = agentcore_runtime.invoke({"prompt": "How is the weather now?"})
invoke_response

In [4]:
import sys
import os

# Get the current notebook's directory
current_dir = os.path.dirname(os.path.abspath("__file__" if "__file__" in globals() else "."))

utils_dir = os.path.join(current_dir, "..")
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

from utils import setup_cognito_user_pool, reauthenticate_user

sys.path[0]: /Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/06-workshops


### Invoking AgentCore Runtime without authorization

Finally, we can invoke our AgentCore Runtime with a payload. Try running the following cell and you will see an error that says **"AccessDeniedException: An error occurred (AccessDeniedException) when calling the InvokeAgentRuntime operation: Agent is configured for a different authorization token type".**

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [13]:
invoke_response = agentcore_runtime.invoke({"prompt": "How is the weather now?"})
invoke_response

## Preparing your agent for deployment on AgentCore Runtime

### Strands Agents with Amazon Bedrock model
Let's start with our Strands Agent we created in the 01-AgentCore-runtime tutorial and configure it with Inbound Auth that uses Amazon Cognito as the Identity Provider.

In [6]:
%%writefile strands_claude.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"


model_id = "global.amazon.nova-2-lite-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

@app.entrypoint
def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

Overwriting strands_claude.py


## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

**Important** - Update the Cognito Discovery url and the Cognito App client id from the previous steps.

In [7]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
region

discovery_url = cognito_config.get("discovery_url")

client_id = cognito_config.get("client_id")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_agent_inbound_identity",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id],
        }
    },
)
response

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Generated Dockerfile: 
/Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/06-workshops/03-AgentCore-identity/03-Inb
ound Auth example/Dockerfile

Generated .dockerignore: /Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/06-workshops/03-AgentCore-identity/03-Inbound Auth example/.dockerignore
Setting 'strands_agent_inbound_identity' as default agent
Bedrock AgentCore configured: /Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/06-workshops/03-AgentCore-identity/03-Inbound Auth example/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/06-workshops/03-AgentCore-identity/03-Inbound Auth example/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/06-workshops/03-AgentCore-identity/03-Inbound Auth example/Dockerfile'), dockerignore_path=PosixPath('/Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/06-workshops/03-AgentCore-identity/03-Inbound Auth example/.dockerignore'), runtime='None', runtime_type=None, region='us-east-1', account_id='402020382644', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

## Review the AgentCore configuration

In [8]:
!cat .bedrock_agentcore.yaml

default_agent: strands_agent_inbound_identity
agents:
  strands_agent_inbound_identity:
    name: strands_agent_inbound_identity
    language: python
    node_version: null
    entrypoint: /Users/pathipat/Work/Learning/Yip-aws-strands&agentcore/agentcore-samples/06-workshops/03-AgentCore-identity/03-Inbound
      Auth example/strands_claude.py
    deployment_type: container
    runtime_type: null
    platform: linux/arm64
    container_runtime: none
    source_path: null
    aws:
      execution_role: null
      execution_role_auto_create: true
      account: '402020382644'
      region: us-east-1
      ecr_repository: null
      ecr_auto_create: true
      s3_path: null
      s3_auto_create: false
      network_configuration:
        network_mode: PUBLIC
        network_mode_config: null
      protocol_configuration:
        server_protocol: HTTP
      observability:
        enabled: true
      lifecycle_configuration:
        idle_runtime_session_timeout: null
        max_lifetime: n

### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [9]:
launch_result = agentcore_runtime.launch()
launch_result

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'strands_agent_inbound_identity' to account 402020382644 (us-east-1)
Generated image tag: 20260805-044912-530
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: strands_agent_inbound_identity
ECR repository available: 402020382644.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-strands_agent_inbound_identity
Getting or creating execution role for agent: strands_agent_inbound_identity
Using AWS region: us-east-1, account ID: 402020382644
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-2213d1079e


✅ Reusing existing ECR repository: 402020382644.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-strands_agent_inbound_identity


✅ Reusing existing execution role: arn:aws:iam::402020382644:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-2213d1079e
Execution role available: arn:aws:iam::402020382644:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-2213d1079e
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: strands_agent_inbound_identity
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-2213d1079e
Reusing existing CodeBuild execution role: arn:aws:iam::402020382644:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-2213d1079e
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: strands_agent_inbound_identity/source.zip
Updated CodeBuild project: bedrock-agentcore-strands_agent_inbound_identity-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.3s
🔄 PROVISIONING started (total: 2s)
✅ PROVISIONING completed in 5.2s
🔄 DO

LaunchResult(mode='codebuild', tag='bedrock_agentcore-strands_agent_inbound_identity:None', env_vars=None, port=None, runtime=None, ecr_uri='402020382644.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-strands_agent_inbound_identity:20260805-044912-530', agent_id='strands_agent_inbound_identity-6rhqbY720O', agent_arn='arn:aws:bedrock-agentcore:us-east-1:402020382644:runtime/strands_agent_inbound_identity-6rhqbY720O', codebuild_id='bedrock-agentcore-strands_agent_inbound_identity-builder:3e97f7d0-978e-4593-8e6c-4d26b6b48256', build_output=None)

### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [12]:
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

Retrieved Bedrock AgentCore status for: strands_agent_inbound_identity


'READY'

### Invoking AgentCore Runtime without authorization

Finally, we can invoke our AgentCore Runtime with a payload. Try running the following cell and you will see an error that says **"AccessDeniedException: An error occurred (AccessDeniedException) when calling the InvokeAgentRuntime operation: Agent is configured for a different authorization token type".**

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [14]:
bearer_token = reauthenticate_user(cognito_config.get("client_id"))
invoke_response = agentcore_runtime.invoke({"prompt": "How is the weather now?"}, bearer_token=bearer_token)
invoke_response

Using JWT authentication


{'response': '"The weather is currently sunny."'}

### Invoking AgentCore Runtime with authorization

Lets invoke the agent with the right authorization token type. In our case, it will be the Cognito access token. Copy the access token from the cell "**Provision a Cognito User Pool**"

In [15]:
bearer_token = reauthenticate_user(cognito_config.get("client_id"))
invoke_response = agentcore_runtime.invoke({"prompt": "How is the weather now?"}, bearer_token=bearer_token)
invoke_response

Using JWT authentication


{'response': '"The weather is currently sunny.\\n\\nWould you like to know the weather for a specific location, or would you like me to check the weather for you again in a little while?"'}

## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [ ]:
from boto3.session import Session
import boto3

boto_session = Session()

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)

# Congratulations!